In [2]:
pip install -U google-genai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from google import genai
from dotenv import load_dotenv
import os
import json

load_dotenv(r"C:\Users\PC\Desktop\100 DAYS OF AI\WEEK 1\DAY 1\.env")

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

In [4]:
paragraph = """
Marie Curie discovered radium. She worked at the University of Paris. 
Her research influenced CERN's later work on radioactivity.
"""

prompt = f"""
Extract relation triples from the following text in the form 
(subject, relation, object).

Return ONLY valid JSON in this format, with no extra text:
[{{"subject": "...", "relation": "...", "object": "..."}}]

Text: {paragraph}
"""

In [5]:
response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=prompt
)

print(response.text)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


[
  {"subject": "Marie Curie", "relation": "discovered", "object": "radium"},
  {"subject": "Marie Curie", "relation": "worked at", "object": "University of Paris"},
  {"subject": "Marie Curie's research", "relation": "influenced", "object": "CERN's later work on radioactivity"}
]


In [6]:
raw_text = response.text.strip()

# Handle both cases: with or without code fences
if raw_text.startswith("```"):
    raw_text = raw_text.split("```")[1]
    raw_text = raw_text.removeprefix("json").strip()

triples = json.loads(raw_text)

for t in triples:
    print(f'{t["subject"]} --{t["relation"]}--> {t["object"]}')

Marie Curie --discovered--> radium
Marie Curie --worked at--> University of Paris
Marie Curie's research --influenced--> CERN's later work on radioactivity


In [8]:
def extract_relations(text):
    prompt = f"""
Extract relation triples from the following text in the form 
(subject, relation, object).

Return ONLY valid JSON in this format, with no extra text:
[{{"subject": "...", "relation": "...", "object": "..."}}]

Text: {text}
"""
    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )
    raw_text = response.text.strip()

    if raw_text.startswith("```"):
        raw_text = raw_text.split("```")[1]
        raw_text = raw_text.removeprefix("json").strip()

    return json.loads(raw_text)


# Test with a new paragraph
test_paragraph = "Larry Page co-founded Google. Google acquired"

conclusion = """
## Conclusion — Day 24: Relation Extraction

Today I learned how to extract relation triples (X → relation → Y) from 
unstructured text using Gemini.

### Key steps:
1. Migrated from the deprecated `google.generativeai` package to the new 
   `google-genai` SDK (Client-based instead of configure/GenerativeModel).
2. Wrote a prompt asking Gemini to extract (subject, relation, object) 
   triples as JSON.
3. Called `client.models.generate_content()` and inspected the raw response.
4. Parsed the response into Python objects using `json.loads()`, handling 
   the case where Gemini wraps output in code fences.
5. Wrapped the flow into a reusable `extract_relations(text)` function 
   and tested it on a new paragraph.

### Why this matters:
Relation triples are the missing piece from Day 23. Entities become 
**nodes**; relation triples become the **directed, labeled edges** 
between them. Together, entity extraction + relation extraction form 
a complete pipeline for building a knowledge graph

In [11]:
from pathlib import Path

readme_lines = [
    "# Day 24 — Relation Extraction",
    "",
    "## Project Overview",
    "",
    "Today I learned how to extract relation triples (subject, relation, object) from unstructured text using Gemini.",
    "",
    "## Concepts Learned",
    "",
    "- What relation extraction is",
    "- The (subject, relation, object) triple structure",
    "- Why direction matters in a relation (connects to directed graphs, Day 22)",
    "- Prompting Gemini for structured triple output as JSON",
    "- Migrating from the deprecated google.generativeai package to the new google-genai SDK",
    "- Wrapping the flow into a reusable extract_relations(text) function",
    "",
    "## Mini-Project",
    "",
    "I extracted relation triples from a paragraph about Marie Curie.",
    "",
    "**Example Output:**",
    "",
    "- Marie Curie --discovered--> radium",
    "- Marie Curie --worked at--> University of Paris",
    "- Marie Curie's research --influenced--> CERN's later work on radioactivity",
    "",
    "I then tested the reusable function on a new paragraph about Larry Page and Google.",
    "",
    "## Technologies Used",
    "",
    "- Python",
    "- google-genai (new Gemini SDK)",
    "- gemini-3.6-flash",
    "- json module",
    "",
    "## Key Learning",
    "",
    "Relation triples are the missing piece after entity extraction (Day 23). Entities become nodes, relation triples become the directed edges between them. Together they form a complete pipeline for building a knowledge graph from raw text — a core building block toward my Graph RAG goal.",
    "",
    "**Day 24 Complete! 🚀**"
]

readme_path = Path(
    r"C:\Users\PC\Desktop\100 DAYS OF AI\WEEK 4\DAY 24\README.md"
)
readme_path.write_text("\n".join(readme_lines), encoding="utf-8")
print("README.md created successfully!")

README.md created successfully!
